In [ ]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Preparación de dataset

In [ ]:
# Cargamos el dataset
df = pd.read_csv('letterboxd_250movie_reviews.csv')

df.rename(columns={"Unnamed: 0":"id"}, inplace=True)

df.head()

,id,Review,Rating,Date,Status,Movie
0,0,I am an idiot. Why is it that I still dread wa...,★★★★★,01 Dec 2013,Watched,harakiri
1,1,honor in the individual is virtue honor in a s...,★★★★★,18 Jan 2022,NaN,harakiri
2,2,Brilliant. Riveting exploration of mortality a...,NaN,26 Jan 2024,Watched,harakiri
3,3,God damn what an impeccable film. Not dated in...,★★★★★,21 Jun 2021,Watched,harakiri
4,4,95First thing's first: giving a big thanks to ...,★★★★★,06 Sep 2020,Watched,harakiri


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90016 entries, 0 to 90015
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      90016 non-null  int64
 1   Review  90011 non-null  str  
 2   Rating  86513 non-null  str  
 3   Date    90016 non-null  str  
 4   Status  86858 non-null  str  
 5   Movie   90016 non-null  str  
dtypes: int64(1), str(5)
memory usage: 4.1 MB


In [ ]:
df["Movie"].values.toList()

AttributeError: 'StringArray' object has no attribute 'toList'

In [ ]:
for column in df.columns:
    print("Valores de " + column + ": ")
    print(df[column].tolist())


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



# Modificaciones
## Valores falatantes de estrellas


# Preguntas
## columna Rating
- ¿Qué valores toma exactamente? ¿Cuántas clases distintas hay?

- ¿Cuántas reseñas hay por cada valor? ¿El dataset está balanceado?

- ¿Se observa un sesgo hacia los puntajes altos? ¿Las medias estrellas aparecen con la misma frecuencia que las estrellas enteras?

- ¿Cómo puede afectar esto al entrenamiento de los modelos y a la métrica que elijan?

- ¿Hay reseñas sin puntaje? ¿Qué harían con ellas?

## Análisis exploratorio inicial del dataset

- ¿Cuántas reseñas contiene? ¿Cuántas películas distintas?

El csv contiene 90015 reseñas en total antes de realizar cualuqier tipo de modificación

- ¿Qué columnas están disponibles?

- ¿Qué tipos de datos tiene cada variable?

- ¿Existen valores faltantes? ¿Y reseñas duplicadas?

Algunas filas tienen valores faltantes en columnas como rating o status.

No existen reseñas duplicadas

- ¿Cuál es la longitud típica de una reseña? ¿Hay reseñas vacías o de una sola
palabra?

# TODO

## Pipeline

- Naive Bayes
- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machines


## Métricas de evaluación

- Ponderar métricas invariantes de la distribución de clases en el dataset para evitar tener que nivelarlo a mano

- Queremos optmimizar precisión general o precisión dentro de cada clase??

### Clasificación multiclase

- Binario por precisión
- Por cercanía de clase

### Regresión

## Separación del dataset

- Aletaroio?
- Nivelados por puntuacion?
- Nivelados por películas que aparecen en cada set?

## Review

- calcular porcentaje de reseñas en otros idiomas y decidir que hacer con ellas...
  - Traducirlas / detectarlas usando una api externa (https://docs.cloud.google.com/translate/docs/reference/rest)
  - Traducirlas usando otro modelo
  - Eliminarlas
- Evaluar si queremos tratar diferentes las filas sin review de corta longitud
- Ver distribución de la longitud de la reseña
- Hacer un boxplots para ver como afectan los diferentes features sobre rating
  - longitud de review
  - emojis
  - uso de mayúsculas (palabras enteras en mayusculas)
  - uso y cantidad de signos de exclamación
  - repeticion de caracteres
  - repeticion de palabras
  - *opcional:* ver como codificar sarcasmo (espacios??)

- Definir como hacemos el procesamiento de texto
  - signos de puntutación
  - nombres propios
  - stop words

## Rating
Teóricamente tomaría 10 valores dicretos (clases) diferentes (0 a 5 estrellas de a 1/2 estrella de paso).

- Chequear que tome todos los valores posibles
- Calcular N° reseñas por valor
- Calcular promedio de longitud de review por valor de rating
- Decidir que hacer con las filas sin reseña (3500/90000)...
  - Eliminarlos
  - Etiquetarlos a mano
  - Asignarles valor según similitud con otras reseñas

## Date
- Revisar la evolución de promedio de rating en el tiempo
- Revisar la evolución del promedio de longitud de review en el tiempo
- *opcional*: Evaluar si el tiempo entre el estreno de la peli y la reseña modifica el rating

## Status
- Anaizar como cambia el promedio (histograma) de ranking según el status (watched, rewatched)

## Movie

- La tenemos en cuenta?
  - Tenerlas en cuenta

In [ ]:
# 1. ¿Cuántas reseñas contiene? ¿Cuántas películas distintas?

# Cantidad de filas y columnas
print("Dimensiones del dataset:", df.shape)

# Cantidad total de registros/reseñas
print("Cantidad de reseñas:", len(df))

# Cantidad de películas distintas
print("Cantidad de películas distintas:", df["Movie"].nunique())

In [ ]:
# 2. ¿Qué columnas están disponibles?

print("Columnas disponibles:")
print(df.columns.tolist())

In [ ]:
# 3. ¿Qué tipos de datos tiene cada variable?

print("Tipos de datos de cada variable:")
print(df.dtypes)

# Información general del dataset

df.info()

In [ ]:
# 4. ¿Existen valores faltantes?

print("Valores faltantes por columna:")
print(df.isnull().sum())

# Porcentaje de valores faltantes por columna

faltantes = pd.DataFrame({
    "Cantidad": df.isnull().sum(),
    "Porcentaje": (df.isnull().sum() / len(df) * 100).round(2)
})

faltantes

In [ ]:
# 5. ¿Existen reseñas duplicadas?

# Primero verificamos si existen filas completamente duplicadas
print("Filas completamente duplicadas:", df.duplicated().sum())

# Luego verificamos específicamente si se repite el texto de alguna review
print("Textos de reviews repetidos:", df.duplicated(subset=["Review"]).sum())

# Mostramos las reviews cuyo texto aparece más de una vez

reviews_repetidas = df[
    df.duplicated(subset=["Review"], keep=False)
].sort_values("Review")

reviews_repetidas.head(20)

In [ ]:
# 6. ¿Cuál es la longitud típica de una reseña?

# Creamos una nueva variable con la cantidad de palabras de cada review
df["review_word_count"] = (
    df["Review"]
    .fillna("")
    .str.split()
    .str.len()
)

# Estadísticas descriptivas de la longitud
df["review_word_count"].describe()

In [ ]:
# 7. ¿Hay reseñas vacías?

reviews_vacias = (
    df["Review"].isna()
    | df["Review"].fillna("").str.strip().eq("")
).sum()

print("Cantidad de reviews vacías:", reviews_vacias)

In [ ]:
# 8. ¿Hay reseñas de una sola palabra?

reviews_una_palabra = (df["review_word_count"] == 1).sum()

print("Cantidad de reviews de una sola palabra:", reviews_una_palabra)

In [ ]:
# 9. ¿Cómo se distribuye la longitud de las reseñas?

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(df["review_word_count"], bins=50)

plt.xlabel("Cantidad de palabras")
plt.ylabel("Cantidad de reviews")
plt.title("Distribución de la longitud de las reviews")

plt.show()

# Distribución de longitud considerando reviews de hasta 500 palabras

plt.figure(figsize=(10, 5))

plt.hist(
    df.loc[df["review_word_count"] <= 500, "review_word_count"],
    bins=50
)

plt.xlabel("Cantidad de palabras")
plt.ylabel("Cantidad de reviews")
plt.title("Distribución de la longitud de las reviews (hasta 500 palabras)")

plt.show()